# 📘 学习注释版：Gold Customer Dimension

**目的：把 CRM Customer + ERP Customer + ERP Location 合成客户维表。**

输出：`workspace.gold.dim_customers`

重点：
- 多表 LEFT JOIN
- `ROW_NUMBER()` 生成 surrogate key
- Dimension 用来描述“客户是谁”


#The Transformation Logic

## 🧩 学习说明：构建 Customer Dimension

**Input：**
- `silver.crm_customers`
- `silver.erp_customers`
- `silver.erp_customer_location`

**Process：**
- LEFT JOIN 多源客户信息
- CASE/字段组合
- `ROW_NUMBER()` 生成 `customer_key`

**Output：** DataFrame `df`，下一步写成 `gold.dim_customers`


In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key,
    ci.customer_id,
    ci.customer_number,
    ci.first_name,
    ci.last_name,
    la.country,
    ci.marital_status,
    CASE
        WHEN ci.gender <> 'n/a' THEN ci.gender
        ELSE COALESCE(ca.gender, 'n/a')
    END AS gender,
    ca.birth_date AS birthdate,
    ci.created_date AS create_date
FROM silver.crm_customers ci
LEFT JOIN silver.erp_customers ca
    ON ci.customer_number = ca.customer_number
LEFT JOIN silver.erp_customer_location la
    ON ci.customer_number = la.customer_number
"""
df = spark.sql(query)


## 👀 学习说明：DataFrame Sanity Check

只显示前 10 行，快速确认当前 DataFrame：
- 字段是否正确
- 清洗是否生效
- 数据是否仍然存在

这一步不写表，只是开发时的中间检查。


In [0]:
df.limit(10).display()

#Writing Gold Table

## 💾 学习说明：把 DataFrame 持久化为 Delta Table

**Input：** 当前 `df`  
**Process：**
- `mode("overwrite")`：目标已存在时覆盖
- `format("delta")`：使用 Delta 格式
- `saveAsTable()`：注册为 Catalog Table

**Output：** `workspace.gold.dim_customers`

注意：这也是为什么 Bootcamp 可以重复运行而通常不会因为“表已存在”直接失败。


In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")

## Sanity checks of Gold table

## ✅ 学习说明：验证 Customer Dimension

确认：
- `customer_key` 已生成
- CRM + ERP + Location 信息已经正确组合
- Gold 表可以供下游分析使用


### 💡 VS Code 阅读版：Databricks SQL（仅展示，不在本地执行）

```sql
SELECT * FROM workspace.gold.dim_customers LIMIT 10
```

> 原可执行版本仍保留在 `01_可执行注释版`。在 Databricks 中请执行那一版。
